In [1]:
# -*- coding: utf-8 -*-

import pandas as pd
import requests
from typing import List, Optional, Dict, Any
from config import api_key
from get_fmp_revenue import to_month_end_safe

def get_fmp_financials_quarterly(
    ticker: str,
    api_key: str,
    fields: List[str],
    limit: int = 40
) -> Optional[pd.DataFrame]:
    """
    FMP API에서 사용자가 지정한 재무 항목들을 분기별로 동적 추출합니다.

    Args:
        ticker: 주식 티커
        api_key: FMP API 키
        fields: 추출할 재무 항목 리스트 (예: ['revenue', 'operatingIncome', 'eps'])
        limit: 가져올 데이터 개수
    """
    # 1. API 호출 (period=quarter 강제)
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}?period=quarter&limit={limit}&apikey={api_key}"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"[Error] API 호출 실패: {e}")
        return None

    if not data:
        return None

    extracted_data = []
    for item in data:
        # 분기 데이터만 필터링
        period = item.get('period', '')
        if period not in ['Q1', 'Q2', 'Q3', 'Q4']:
            continue

        # 공통 기본 정보 설정
        row = {
            'ticker': ticker,
            'date': item.get('date'),
            'date_month_end': to_month_end_safe(pd.Series([item.get('date')]))[0],
            'calendar_year': item.get('calendarYear'),
            'period': period
        }

        # 사용자가 요청한 필드를 동적으로 추가
        for field in fields:
            # FMP 데이터는 보통 원단위이므로 10억 단위(Billions)로 변환
            value = item.get(field, 0)
            if value is None: value = 0

            # 수치형 데이터인 경우 Billions 변환 (필드명에 _billions 접미사 추가)
            row[f"{field}_billions"] = round(value / 1_000_000_000, 3)

        extracted_data.append(row)

    df = pd.DataFrame(extracted_data)

    # 날짜 순으로 정렬 및 중복 제거
    if not df.empty:
        df = df.sort_values('date_month_end').reset_index(drop=True)

    return df

# --- 메인 실행 예시 ---
# if __name__ == "__main__":
#     target_ticker = "AAPL"
#
#     # 사용자가 원하는 항목을 리스트로 정의 (하드코딩 탈피)
#     # FMP Income Statement API의 JSON 키값을 그대로 사용하면 됩니다.
#     desired_fields = ['revenue', 'operatingIncome', 'netIncome', 'ebitda']
#
#     quarterly_df = get_fmp_financials_quarterly(
#         ticker=target_ticker,
#         api_key=api_key,
#         fields=desired_fields
#     )
#
#     if quarterly_df is not None:
#         print(f"\n[*] {target_ticker} 분기별 재무 데이터 (항목: {desired_fields})")
#         print(quarterly_df.tail())

In [19]:

target_ticker = "MU"

# 사용자가 원하는 항목을 리스트로 정의 (하드코딩 탈피)
# FMP Income Statement API의 JSON 키값을 그대로 사용하면 됩니다.
# desired_fields = ['revenue', 'operatingIncome', 'netIncome', 'ebitda']

desired_fields = ['revenue']

quarterly_df = get_fmp_financials_quarterly(
    ticker=target_ticker,
    api_key=api_key,
    fields=desired_fields
)

if quarterly_df is not None:
    print(f"\n[*] {target_ticker} 분기별 재무 데이터 (항목: {desired_fields})")
    print(quarterly_df.tail())


[*] MU 분기별 재무 데이터 (항목: ['revenue'])
   ticker        date date_month_end calendar_year period  revenue_billions
35     MU  2024-11-28     2024-11-30          2025     Q1             8.709
36     MU  2025-02-27     2025-02-28          2025     Q2             8.053
37     MU  2025-05-29     2025-05-31          2025     Q3             9.301
38     MU  2025-08-28     2025-08-31          2025     Q4            11.315
39     MU  2025-11-27     2025-11-30          2026     Q1            13.643


In [20]:
quarterly_df

,ticker,date,date_month_end,calendar_year,period,revenue_billions
0,MU,2016-03-03,2016-03-31,2016,Q2,2.934
1,MU,2016-06-02,2016-06-30,2016,Q3,2.898
2,MU,2016-09-01,2016-09-30,2016,Q4,3.217
3,MU,2016-12-01,2016-12-31,2017,Q1,3.970
4,MU,2017-03-02,2017-03-31,2017,Q2,4.648
5,MU,2017-06-01,2017-06-30,2017,Q3,5.566
6,MU,2017-08-31,2017-08-31,2017,Q4,6.138
7,MU,2017-11-30,2017-11-30,2018,Q1,6.803
8,MU,2018-03-01,2018-03-31,2018,Q2,7.351
9,MU,2018-05-31,2018-05-31,2018,Q3,7.797


In [14]:
# 제공된 모듈 임포트
from DATA.universal_ts_forecast_function import forecast_one_from_pivot_inline, infer_freq_alias

def run_forecast_pipeline(ticker: str, target_field: str, n_step: int = 4):
    """
    1. FMP에서 데이터 추출
    2. 시계열 예측 수행 (universal_ts_forecast_function 활용)
    3. 결과 병합 및 반환
    """
    # [1] 데이터 수집 (사용자 입력 필드 반영)
    # field 입력 시 'revenue'만 넣어도 내부적으로 'revenue_billions'로 생성됨을 고려
    raw_field = target_field.replace("_billions", "")
    df = get_fmp_financials_quarterly(ticker, api_key, fields=[raw_field], limit=100)

    if df is None or df.empty:
        print(f"[오류] {ticker}의 데이터를 가져올 수 없습니다.")
        return None

    # 생성된 컬럼명 확인 (예: revenue_billions)
    col_name = f"{raw_field}_billions"

    # [2] 예측용 Pivot 생성 및 데이터 정제
    # 인덱스를 날짜로 설정
    df['date_month_end'] = pd.to_datetime(df['date_month_end'])
    df_ts = df.set_index('date_month_end').sort_index()

    # 중요: 결측치(NaN)가 있는 행을 제거하여 '연속 구간'을 확보합니다.
    df_ts = df_ts.dropna(subset=[col_name])

    # Pivot 형태로 변환 (함수 요구사항)
    pivot_df = df_ts.pivot(columns='ticker', values=col_name)

    # [3] 시계열 예측 수행
    print(f"[*] {ticker}의 {col_name} 항목 예측 시작 (n_step={n_step})...")

    # 데이터 길이가 짧을 경우를 대비해 모델을 'ETS'나 'Theta'로 병행 검토 가능
    # 여기서는 요청하신 대로 수행하되, 에러 발생 시 핸들링
    forecast_results = forecast_one_from_pivot_inline(
        pivot_df=pivot_df,
        target_col=ticker,
        horizon=n_step,
        models=["SARIMA", "ETS"], # SARIMA 실패 시 대비해 ETS 추가 권장
        strict_no_nan=False
    )

    # 결과 추출 (우선순위: SARIMA -> ETS)
    model_used = None
    fc_values = None

    for m in ["SARIMA", "ETS"]:
        if m in forecast_results and "forecast" in forecast_results[m]:
            fc_values = forecast_results[m]["forecast"]
            model_used = m
            break

    if fc_values is None:
        print(f"[실패] 모든 모델이 예측에 실패했습니다. (데이터 부족 등)")
        return df_ts[[col_name, 'ticker']].reset_index()

    # [4] 미래 날짜 생성 및 결합
    freq = infer_freq_alias(pivot_df.index)
    last_date = pivot_df.index.max()
    forecast_dates = pd.date_range(start=last_date, periods=n_step + 1, freq=freq)[1:]

    # 기존 데이터 정리
    actual_df = df_ts[[col_name]].reset_index()
    actual_df.columns = ['date', 'value']
    actual_df['status'] = 'Actual'

    # 예측 데이터 정리
    forecast_df = pd.DataFrame({
        'date': forecast_dates,
        'value': fc_values,
        'status': f'Forecast({model_used})'
    })

    # 최종 병합
    combined_df = pd.concat([actual_df, forecast_df], ignore_index=True)
    combined_df['ticker'] = ticker
    combined_df.rename(columns={'value': col_name}, inplace=True)

    return combined_df

In [16]:
ticker_input = "APH"
field_input = "revenue" # 또는 "operatingIncome"
steps = 4

final_df = run_forecast_pipeline(ticker_input, field_input, n_step=steps)

[*] APH의 revenue_billions 항목 예측 시작 (n_step=4)...
[메모리] forecast_one_from_pivot_inline 실행 전: 382.70 MB

[예측 중] SARIMA 모델...
[메모리] forecast_sarima 실행 전: 382.73 MB
[메모리] find_best_sarima_params 실행 전: 382.73 MB

[메모리] find_best_sarima_params 실행 후: 388.89 MB (변화: +6.16 MB)
[메모리] forecast_sarima 실행 후: 388.93 MB (변화: +6.20 MB)

[예측 중] ETS 모델...
[메모리] forecast_ets 실행 전: 388.97 MB
[메모리] forecast_ets 실행 후: 389.16 MB (변화: +0.19 MB)
[메모리] forecast_one_from_pivot_inline 실행 후: 389.16 MB (변화: +6.46 MB)


In [18]:
final_df

,date,revenue_billions,status,ticker
0,2000-12-31,0.369000,Actual,APH
1,2001-03-31,0.317000,Actual,APH
2,2001-06-30,0.274000,Actual,APH
3,2001-09-30,0.253000,Actual,APH
4,2001-12-31,0.260000,Actual,APH
...,...,...,...,...
99,2025-09-30,6.194000,Actual,APH
100,2025-12-31,6.627622,Forecast(SARIMA),APH
101,2026-03-31,7.045730,Forecast(SARIMA),APH
102,2026-06-30,7.810317,Forecast(SARIMA),APH
